In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import gc
import time
#import matplotlib.pyplot as plt
#from livelossplot.keras import PlotLossesCallback
from tensorflow.keras.callbacks import ReduceLROnPlateau
#from keras.models import load_model
import csv

In [7]:
data_train=pd.read_csv('F:/TensorFlow/traindata20210625.csv',header=0,encoding='gbk')
data_train.info()
data_train.Value.unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Columns: 262 entries, FID to Wgs84Y
dtypes: float64(26), int64(233), object(3)
memory usage: 26.6+ MB


array([9991, 4972,  415,  510, 4991,  498,  416, 9995, 4261, 5045,  246,
       4971, 4831, 5061, 4871,  286,   12, 5081,  560, 5083,  535, 1995,
       4792,  518,  342, 5062,  308,  349,  503,  534,  380, 4491,  394,
       2462, 9993,  288,  305,  338,   14,  328,  278, 4492, 9994,  267,
       3281,  275, 5113,   85, 4264,  360, 4841, 2473,   89,   91,  197,
        370,  312,  241,   62,  376,  287,  513,  892,  273,  387,   15,
        516,  472, 5042,    0,  374, 4951,  419, 3751,   94, 5114,  202,
       3711,  285, 9992, 4151,  401,  425, 2432,  395, 5112,  368,  557,
        561,    4,  512,  351, 4111,  529,  477,  533,  538,  249, 5071,
       1831, 2431,  539,  537, 5032, 5141, 4011, 2461, 5111, 3713, 2472,
       5461,  429, 1961, 5131,  536,  257], dtype=int64)

In [8]:
pd.set_option('display.max_rows', None)
print(data_train['Value'].value_counts())
#aaa.to_csv('20200627counts.csv', index=True)

4972    4583
498     1197
5061    1161
415      729
4991     484
5045     429
360      317
4261     316
370      288
376      285
513      205
516      205
246      175
2462     166
419      162
387      151
534      151
561      136
4831     107
5032     103
5083     102
9995      99
5081      82
510       76
5042      71
4841      60
472       56
342       55
538       50
503       50
380       49
557       49
267       48
349       44
4792      40
3711      40
537       36
2431      35
5141      34
512       33
560       33
533       33
5113      32
9993      29
5111      29
305       28
9992      27
394       26
2461      26
4264      25
273       25
4151      24
85        23
539       21
374       21
308       19
518       16
9991      16
1961      16
338       15
4491      15
416       15
395       15
4971      15
241       14
4111      14
5114      13
1831      13
4         13
3751      12
3281      11
328       11
62        11
287       10
536       10
5071      10
4011       9

In [9]:
data_train.head()

,FID,序号,校正_群,中国植,大类,植被代,Value,AET1901,AET1902,AET1903,...,SR190829B7,SR190930B1,SR190930B2,SR190930B3,SR190930B4,SR190930B5,SR190930B6,SR190930B7,Wgs84X,Wgs84Y
0,0,4260,高山流石坡稀疏植被,高山流石坡稀疏植被,高山流石坡稀疏植被,9991,9991,0,0,0,...,2610,1462,2263,806,1153,3118,3278,2726,98.279183,39.040367
1,1,4261,高山流石坡稀疏植被,高山流石坡稀疏植被,高山流石坡稀疏植被,9991,9991,0,0,0,...,2610,1462,2263,806,1153,3118,3278,2726,98.279183,39.040367
2,2,4262,高山流石坡稀疏植被,高山流石坡稀疏植被,高山流石坡稀疏植被,9991,9991,0,0,0,...,2610,1462,2263,806,1153,3118,3278,2726,98.279183,39.040367
3,3,4263,高山流石坡稀疏植被,高山流石坡稀疏植被,高山流石坡稀疏植被,9991,9991,0,0,0,...,2610,1462,2263,806,1153,3118,3278,2726,98.279183,39.040367
4,4,4231,高山嵩草草甸,高山嵩草（Kobresia pygmaea）草甸,草甸,0,4972,0,0,0,...,1117,864,1824,457,699,2716,2402,1444,98.808435,38.985385


In [10]:
# 生成该区间的随意唯一索引
index = np.random.permutation(len(data_train))
# 用生成的乱的索引就能将其打乱了
data_train = data_train.iloc[index ,:]

In [11]:
train_x = data_train.iloc[:,7:]
train_y = data_train.loc[:, ['Value']]  
train_x.head(), train_y.head()

(       AET1901  AET1902  AET1903  AET1904  AET1905  AET1906  AET1907  AET1908  \
 6320         0        0        0      652      875      880     1020      930   
 8731         0        0        0        0      294      868      977      958   
 10900        0       36       27      754      918     1003     1040      980   
 12325        0        0        0      319       89      102      663      403   
 3636         0        0        0      647      752      937      844     1116   
 
        AET1909  AET1910  ...  SR190829B7  SR190930B1  SR190930B2  SR190930B3  \
 6320       733      416  ...        1163         690        2064         258   
 8731       700        0  ...        1332         121         484         304   
 10900      798      555  ...         967         942        3436         338   
 12325      274       90  ...        2260        1842        2569         874   
 3636       727      214  ...         738         614        1868         391   
 
        SR190930B4

In [12]:
data_train['Value'].nunique()

116

In [13]:
train_y=tf.keras.utils.to_categorical(train_y)

In [14]:
train_y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [15]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(4096,input_dim=255))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dense(2048,activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dense(1024,activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dense(512,activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dense(256,activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dense(128,activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dense(64,activation='relu'))
model.add(tf.keras.layers.Dropout(rate=0.5))
model.add(tf.keras.layers.Dense(116, activation='softmax'))
# 让学习率自动变化
reduce_lr = ReduceLROnPlateau(monitor='loss',factor=0.1, patience=20, mode='auto')
model.compile(loss='categorical_crossentropy', metrics=['acc'], optimizer='adam')#adam优化
#顺序编码用loss='sparse_categorical_crossentropy'
#独热编码用loss='categorical_crossentropy'
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense (Dense)                (None, 4096)              1048576   
_________________________________________________________________
batch_normalization (BatchNo (None, 4096)              16384     
_________________________________________________________________
dense_1 (Dense)              (None, 2048)              8390656   
_________________________________________________________________
batch_normalization_1 (Batch (None, 2048)              8192      
_________________________________________________________________
dense_2 (Dense)              (None, 1024)              2098176   
_________________________________________________________________
batch_normalization_2 (Batch (None, 1024)              4096      
_________________________________________________________________
dense_3 (Dense)              (None, 512)               5

In [16]:
e = 200
start = time.perf_counter()#time.process_time()测的时间偏长
history = model.fit(train_x,train_y, epochs=e,batch_size=16,verbose=1,callbacks=[reduce_lr])#batch_size=16
mins, secs = divmod((time.perf_counter() - start), 60)
hours, mins = divmod(mins, 60)
print("用时：%02d:%02d:%02d"% (hours, mins, secs))

ValueError: A target array with shape (13320, 9996) was passed for an output of shape (None, 116) while using as loss `categorical_crossentropy`. This loss expects targets to have the same shape as the output.

In [ ]:
#gc.collect()
#def predict(file):
#    #mylist = []
#    index=0
#    for chunk in pd.read_csv(file, chunksize=500000):
#        print(index)
#        rchunk = pd.DataFrame(model.predict_classes(chunk.iloc[:,1:],verbose=1))
#        #mylist.append(rchunk)
#        arry1 = np.array(rchunk) #先转成np数据
#        list1 = arry1.tolist()  #再转成list
#        with open('/content/drive/My Drive/DL20200709/testresult20200728_p1.csv','a',newline='',encoding='utf-8') as f:
#          writer = csv.writer(f)
#          writer.writerows(list1)
#        index+=1
#    #temp_df = pd.concat(mylist, axis= 0)
#    #del mylist
#    #return temp_df
#start = time.perf_counter()
#f = open('/content/drive/My Drive/DL20200709/testdata20200706_p1.csv',encoding='utf-8')#p2 ,encoding='gb2312'
#prediction = predict(f)
##np.savetxt("testresult20200523caoyuan.csv", prediction, fmt="%d",delimiter=",") 
#mins, secs = divmod((time.perf_counter() - start), 60)
#hours, mins = divmod(mins, 60)
#print("用时：%02d:%02d:%02d"% (hours, mins, secs))
#p2

gc.collect()
def predict(file):
    #mylist = []
    index=0
    for chunk in pd.read_csv(file, chunksize=500000):
        print(index)
        rchunk = pd.DataFrame(model.predict_classes(chunk.iloc[:,1:],verbose=1))
        #mylist.append(rchunk)
        arry1 = np.array(rchunk) #先转成np数据
        list1 = arry1.tolist()  #再转成list
        with open('/content/drive/My Drive/DL20200709/testresult20200728_p2.csv','a',newline='',encoding='utf-8') as f:
          writer = csv.writer(f)
          writer.writerows(list1)
        index+=1
    #temp_df = pd.concat(mylist, axis= 0)
    #del mylist
    #return temp_df

start = time.perf_counter()
f = open('/content/drive/My Drive/DL20200709/testdata20200706_p22.csv',encoding='gb2312')#p2 ,encoding='gb2312'
prediction = predict(f)
#np.savetxt("testresult20200523caoyuan.csv", prediction, fmt="%d",delimiter=",") 
mins, secs = divmod((time.perf_counter() - start), 60)
hours, mins = divmod(mins, 60)
print("用时：%02d:%02d:%02d"% (hours, mins, secs))
#p3

gc.collect()
def predict(file):
    #mylist = []
    index=0
    for chunk in pd.read_csv(file, chunksize=500000):
        print(index)
        rchunk = pd.DataFrame(model.predict_classes(chunk.iloc[:,1:],verbose=1))
        #mylist.append(rchunk)
        arry1 = np.array(rchunk) #先转成np数据
        list1 = arry1.tolist()  #再转成list
        with open('/content/drive/My Drive/DL20200709/testresult20200728_p3.csv','a',newline='',encoding='utf-8') as f:
          writer = csv.writer(f)
          writer.writerows(list1)
        index+=1
    #temp_df = pd.concat(mylist, axis= 0)
    #del mylist
    #return temp_df

start = time.perf_counter()
f = open('/content/drive/My Drive/DL20200709/testdata20200706_p3.csv',encoding='utf-8')#p2 ,encoding='gb2312'
prediction = predict(f)
#np.savetxt("testresult20200523caoyuan.csv", prediction, fmt="%d",delimiter=",") 
mins, secs = divmod((time.perf_counter() - start), 60)
hours, mins = divmod(mins, 60)
print("用时：%02d:%02d:%02d"% (hours, mins, secs))

0
Instructions for updating:
Please use instead:* `np.argmax(model.predict(x), axis=-1)`,   if your model does multi-class classification   (e.g. if it uses a `softmax` last-layer activation).* `(model.predict(x) > 0.5).astype("int32")`,   if your model does binary classification   (e.g. if it uses a `sigmoid` last-layer activation).
15625/15625 [==============================] - 29s 2ms/step
1
15625/15625 [==============================] - 30s 2ms/step
2
15625/15625 [==============================] - 30s 2ms/step
3
15625/15625 [==============================] - 28s 2ms/step
4
15625/15625 [==============================] - 28s 2ms/step
5
15625/15625 [==============================] - 27s 2ms/step
6
15625/15625 [==============================] - 27s 2ms/step


OSError: ignored